# 03. Development-only Random Forest tuning

Run the indices-only comparison in Testing 03 first. This notebook tunes the 72-predictor fixed and aligned matched forests. It holds the predictors and samples fixed and compares the original settings with four one-setting changes. Five geographic folds within development training determine the selected settings. Locked validation provides a separate check before external scoring.


In [ ]:
from pathlib import Path
import sys

root = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "pyproject.toml").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from crop_pipeline.training import model_comparisons, model_followups

training_run = root / "pipeline_outputs/training/run_fa40f5d6bb31"
testing_run = root / "pipeline_outputs/testing/run_81a0d0a2812e"
batch2_output = root / "pipeline_outputs/testing/run_81a0d0a2812e_batch2"
comparators = batch2_output / "comparators.json"


## Select on development and freeze

Use the original spatial folds and weights. Rank candidates by pooled out-of-fold parcel macro-F1. Exact ties favor the original settings, then the recorded candidate order. The completed receipt allows later runs to reuse the fitted models.


In [ ]:
tuned_receipt = batch2_output / "tuned_comparison" / "comparators.json"
if not tuned_receipt.is_file():
    tuned_receipt = model_followups.freeze_followup(training_run, comparators, "tuned")
print("Frozen tuned-model receipt:", tuned_receipt)


## Compare on the shared external parcels

Compare with the original forests on identical parcels using the revised Batch 1 scope. The paired interval resamples whole parcels within campaign and reference crop and uses the same draws for both forests.


In [ ]:
from IPython.display import Markdown, display

tuned_scores = model_comparisons.score_comparators(tuned_receipt, testing_run)
tuned_report = model_followups.render_followup(tuned_receipt, tuned_scores)
display(Markdown(tuned_report.read_text(encoding="utf-8")))
print("Tuning results:", tuned_receipt.parent)
